## Validation of model_predict_coupler_NCap_cap_matrix wtih Ansys Q3D

In [11]:
from squadds.simulations.objects import run_capn_LOM 
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee
from qiskit_metal import designs, Dict, MetalGUI
import pandas as pd

Typically Ansys simulations through SQuADDS are handled directly through the ```squadds.AnsysSimulator``` class. However, the functionality of that class is mainly for LOM simulations of single qubits or LOM + eigenmode simulations of qubits + cavity systems. Here we are interested purely in extracting the capacitance matrix of an isolated interdigitated finger capacitor. In order to simulate this via SQuADDS API, we need to dig into the source code a bit.

```run_capn_LOM``` from ```squadds.simulations.object``` is an internally called function used to simulate qubit + cavity system with a half wavelength resonator. Typically, it is used to extract the capacitance matrix of the interdigitated finger capacitor that couples the $\lambda/2$ resonator to the feedline, the results are then used to calculate the coupled resonant frequency and linewidth of the resonator. We call it directly below to simulate stand alone coupling capacitors.

# Read in ML results and get design / simulation setup ready

In [22]:
## read in ML results
df = pd.read_csv("")

'''read in SQuADDS coupler-NCap-cap_matrix dataset'''
ncap_db = pd.read_json("coupler-NCap-cap_matrix.json")

''' design and simulation setup parameters from DB '''
sim_setup = ncap_db.sim_options.iloc[0]
SQuADDS_design_params = ncap_db.design.iloc[0]["design_options"] # substitute ML predicted design params into here 

In [ ]:
results = pd.DataFrame({"Sample":[],
                   "ref_cap_matrix":[],
                   "pred_cap_matrix":[]})

um = 10**6 ## ML model is trained in SI units (m), convert back to µm  
samples_to_test = np.unique(ML_results.sample_idx)

## Sweep through ML design results, simulate each design with Anysys Q3D, and save sim results
There are four multi-valued parameters in the SQuADDS coupler-NCap-cap_matrix dataset:
* design_options.cap_gap
* design_options.cap_width
* design_options.finger_length
* design_options.finger_count
  
The model only predicts these four design parameters. We substitute in the predicted parameters from the model direcly into Quantum (Qiskit) Metal and then simulate the design.

In [ ]:
for sample in samples_to_test:

    ''' current testing sample '''
    this_device = ML_results[ML_results.sample_idx == sample]

    ''' extract predicted design parameters '''
    for param in np.unique(this_device.param):
        param_key = param.split(".")[1]
        design_value = this_device[this_device.param == param].pred_unscaled.iloc[0]
        
        if param_key == 'finger_count':
            SQuADDS_design_params[param_key] = np.round(design_value,0) 
        else:
            SQuADDS_design_params[param_key] = str(design_value*um)+"um"
        
    ''' create ML predicted design in Quantum (Qiskit) Metal '''
    design = designs.DesignPlanar()
    cplr = CapNInterdigitalTee(design,"NCap_{}".format(sample),options = SQuADDS_design_params)
    design.rebuild()

    ''' simulate predicted design '''
    pred_ansys_results = run_capn_LOM(design,cplr.options,default_lom_options)

    pred_cap_matrix = pred_ansys_results["sim_results"]

    # rename the keys to match the simulations keys to simplify analysis later
    ref_cap_matrix = {'C_top2top': this_device.top_to_top.iloc[0],
                      'C_top2bottom': this_device.top_to_bottom.iloc[0],
                      'C_top2ground': this_device.top_to_ground.iloc[0],
                      'C_bottom2bottom': this_device.bottom_to_bottom.iloc[0],
                      'C_bottom2ground': this_device.bottom_to_ground.iloc[0],
                      'C_ground2ground': this_device.ground_to_ground.iloc[0]}
    
    ''' save results '''
    results.loc[len(results)] = [sample,
                                 pred_cap_matrix,
                                 pred_cap_matrix]

# Save the data

In [14]:
results.to_csv("NCap_simulation_results.csv",index=False) # save data for later analysis with validation_01_data_analysis.ipynb